In [4]:
!pip install playwright
!playwright install
!playwright install-deps

Playwright Host validation warning: 
╔══════════════════════════════════════════════════════╗
║ Host system is missing dependencies to run browsers. ║
║ Please install them with the following command:      ║
║                                                      ║
║     playwright install-deps                          ║
║                                                      ║
║ Alternatively, use apt:                              ║
║     apt-get install libxcomposite1\                  ║
║         libgtk-3-0\                                  ║
║         libatk1.0-0                                  ║
║                                                      ║
║ <3 Playwright Team                                   ║
╚══════════════════════════════════════════════════════╝
    at validateDependenciesLinux (/usr/local/lib/python3.12/dist-packages/playwright/driver/package/lib/server/registry/dependencies.js:269:9)
    at async Registry._validateHostRequirements (/usr/local/lib/python3.12/dist

In [5]:
import asyncio
from playwright.async_api import async_playwright

In [22]:
import re

def solve_question(text):

    text = text.replace("What is", "").replace("?", "").strip()

    match = re.search(r'(\d+)\s*([\+\-\*/])\s*(\d+)', text)

    if not match:
        raise ValueError(f"Could not parse question: {text}")

    a = int(match.group(1))
    op = match.group(2)
    b = int(match.group(3))

    if op == "+":
        return a + b
    elif op == "-":
        return a - b
    elif op == "*":
        return a * b
    elif op == "/":
        return int(a / b)

In [37]:
async def math_challenge():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()

        # Clear cookies, localStorage, and permissions to start fresh
        await context.clear_cookies()
        await context.clear_permissions()

        page = await context.new_page()

        # Step 1: Go to website
        await page.goto("https://mathspeedtest.com/")

        # Step 2: Enter username
        await page.fill("input[placeholder='Enter your name']", "Mehedi")

        # Step 3: Click 'Start Test'
        await page.get_by_role("button", name="Start Test").click()
        await page.wait_for_timeout(1000)

        # Step 4: Wait until question page loads
        await page.wait_for_url("**/questions/**")

        # Step 5: Solve 20 questions
        for i in range(1, 21):
            question_locator = f"#question_{i} p"
            question_text = await page.locator(question_locator).inner_text()
            answer = solve_question(question_text)

            # Fill answer
            await page.locator(f"#question_{i} input").fill(str(answer))

            # Move to next question by pressing Enter
            await page.keyboard.press("Enter")

            # Click submit on the 20th question
            if i == 20:
                await page.click("input[type='submit']")

        # Step 6: Wait for results page
        await page.wait_for_url("**/results-page/**")
        print("Reached results page!")

        # Step 7: Wait for the current position row
        await page.wait_for_selector("tr.highlight", timeout=15000)

        # Locate the highlighted row and extract data
        row = page.locator("tr.highlight")
        rank = await row.locator("td").nth(0).inner_text()
        name = await row.locator("td").nth(1).inner_text()
        time_taken = await row.locator("td").nth(2).inner_text()

        print("Rank:", rank)
        print("Name:", name)
        print("Time:", time_taken)

In [38]:
await math_challenge()

Reached results page!
Rank: 12
Name: Nuutti
Time: 37.988
